# Sequential Baseline: Simple Recurrent Neural Network (RNN)
**CSE 4122 — Natural Language Processing Laboratory**  
*Department of Computer Science and Engineering, Khulna University of Engineering & Technology (KUET)*

---

### Overview
This notebook implements a sequential deep learning model for sarcasm detection:
- **Tokenization**: Custom word-level vocabulary building with frequency thresholding and padding.
- **Model Architecture**: Embedding layer $\rightarrow$ Vanilla Recurrent Neural Network (`nn.RNN`) $\rightarrow$ Dropout $\rightarrow$ Fully Connected classification layer.
- **Class Imbalance Handling**: Weighted Cross-Entropy Loss to counter the dataset skew.
- **Evaluation**: Accuracy, Precision, Recall, F1, Loss curves, and decision threshold analysis.


## 1. Setup & Environment

In [ ]:
import os
import re
import json
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] Using device: {device}")


## 2. Dataset Loading

In [ ]:
train_path = os.path.join("dataset", "train.csv")
test_path = os.path.join("dataset", "test_1.csv")

if not os.path.exists(train_path):
    train_path = "train.csv"
if not os.path.exists(test_path):
    test_path = "test_1.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

t_col_train = "tweet" if "tweet" in train_df.columns else "text"
t_col_test = "tweet" if "tweet" in test_df.columns else "text"

def clean_seq_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_texts = [clean_seq_text(t) for t in train_df[t_col_train]]
train_labels = train_df["sarcastic"].astype(int).tolist()

test_texts = [clean_seq_text(t) for t in test_df[t_col_test]]
test_labels = test_df["sarcastic"].astype(int).tolist()

print(f"Loaded {len(train_texts)} train samples, {len(test_texts)} test samples.")


## 3. Vocabulary Builder & Sequence Dataset
Build a word-level tokenizer with `<PAD>` (index 0) and `<UNK>` (index 1).

In [ ]:
class WordTokenizer:
    def __init__(self, max_vocab: int = 5000, max_len: int = 64):
        self.max_vocab = max_vocab
        self.max_len = max_len
        self.vocab = {"<PAD>": 0, "<UNK>": 1}
        self.inv_vocab = {0: "<PAD>", 1: "<UNK>"}

    def fit(self, texts):
        counter = Counter()
        for t in texts:
            words = re.findall(r'\b\w+\b', t.lower())
            counter.update(words)
        
        most_common = counter.most_common(self.max_vocab - 2)
        for idx, (word, _) in enumerate(most_common, start=2):
            self.vocab[word] = idx
            self.inv_vocab[idx] = word

    def encode(self, text):
        words = re.findall(r'\b\w+\b', text.lower())
        ids = [self.vocab.get(w, 1) for w in words]
        if len(ids) < self.max_len:
            ids = ids + [0] * (self.max_len - len(ids))
        else:
            ids = ids[:self.max_len]
        return ids

tokenizer = WordTokenizer(max_vocab=6000, max_len=64)
tokenizer.fit(train_texts)
print(f"Vocabulary size: {len(tokenizer.vocab)}")

class SarcasmSeqDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        seq = self.tokenizer.encode(self.texts[idx])
        return torch.tensor(seq, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

train_dataset = SarcasmSeqDataset(train_texts, train_labels, tokenizer)
test_dataset = SarcasmSeqDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


## 4. Simple RNN Model Architecture

In [ ]:
class SarcasmRNN(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 128, num_classes: int = 2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity='relu')
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        # Take hidden state of last step
        last_hidden = hidden[-1]
        out = self.dropout(last_hidden)
        logits = self.fc(out)
        return logits

model = SarcasmRNN(vocab_size=len(tokenizer.vocab), embed_dim=128, hidden_dim=128).to(device)
print(model)


## 5. Model Training with Class-Weighted Loss
Due to class imbalance (~3:1), standard unweighted loss causes vanilla RNN to collapse into predicting all non-sarcastic labels. We supply inverse class weights.

In [ ]:
# Compute class weights
neg_count = train_labels.count(0)
pos_count = train_labels.count(1)
weight_for_0 = len(train_labels) / (2.0 * neg_count)
weight_for_1 = len(train_labels) / (2.0 * pos_count)
class_weights = torch.tensor([weight_for_0, weight_for_1], dtype=torch.float).to(device)
print(f"Class weights: {class_weights.tolist()}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs = 8
history = {"loss": []}

print("Starting Simple RNN Training...")
model.train()
for epoch in range(num_epochs):
    running_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(train_loader)
    history["loss"].append(epoch_loss)
    print(f"Epoch {epoch+1}/{num_epochs} — Loss: {epoch_loss:.4f}")

print("[OK] Training completed.")


## 6. Training Loss Curve Visualization

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, num_epochs + 1), history["loss"], marker="o", color="royalblue", lw=2)
plt.title("Simple RNN Training Loss Convergence")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


## 7. Test Evaluation & Confusion Matrix

In [ ]:
model.eval()
all_preds = []
all_probs = []
all_targets = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        logits = model(x_batch)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_targets.extend(y_batch.numpy())

all_targets = np.array(all_targets)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

acc = accuracy_score(all_targets, all_preds)
prec, rec, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average="binary", zero_division=0)

print("="*45)
print("           Simple RNN Test Metrics")
print("="*45)
print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall:     {rec:.4f}")
print(f"F1 Score:   {f1:.4f}")
print("="*45)
print("\nClassification Report:\n", classification_report(all_targets, all_preds, target_names=["Non-Sarcastic", "Sarcastic"]))

# Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=["Non-Sarcastic", "Sarcastic"],
            yticklabels=["Non-Sarcastic", "Sarcastic"])
plt.title("Simple RNN Test Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()


## 8. Decision Threshold Optimization
RNN outputs can be calibrated by scanning decision thresholds to balance precision vs recall on the test set.

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
thresh_results = []

for t in thresholds:
    p_t = (all_probs >= t).astype(int)
    acc_t = accuracy_score(all_targets, p_t)
    pr_t, rc_t, f1_t, _ = precision_recall_fscore_support(all_targets, p_t, average="binary", zero_division=0)
    thresh_results.append({"Threshold": t, "Accuracy": acc_t, "Precision": pr_t, "Recall": rc_t, "F1": f1_t})

pd.DataFrame(thresh_results).round(4)
